In [1]:
setwd("/home/user/data3/lit/project/sORFs/06-RNA-seq")
fc_output_file <- "/home/user/data3/lit/project/sORFs/06-RNA-seq/02-output-custom-fa-gtf-20251022/featureCounts/rna-seq-counts.txt"
libsize_file <- "./02-output-custom-fa-gtf-20251022/expr/sample_name.lib.txt"
out_file="./02-output-custom-fa-gtf-20251022/expr/rpkm_N_C_A.txt"

In [13]:
#!/usr/bin/env Rscript
# 功能：从 featureCounts 输出计算 RPKM，并比较核内（N）和核外（C）基因表达差异
source("/home/user/data2/lit/bin/lit_utils.R")
source("/home/user/data3/lit/project/sORFs/sORFs.utils.R")
lib_text()
lib_plot()
###### libsize为UNIQUELY MAPPED READS而不是featureCounts assigned reads ######
fread_c(fc_output_file)  -> rna_counts
libsize <- fread_c(libsize_file)
colnames(libsize) <- c("Sample","libsize")
libsize

# 重命名列
colnames(rna_counts[,7:ncol(rna_counts)]) %>% sub("/home/user/data3/lit/project/sORFs/06-RNA-seq/02-output-20250621/mapping/","",.) %>% 
  sub(".R1_Aligned.sortedByCoord.out.bam","",.) -> colnames_organized
head(colnames_organized)

colnames_organized_1 <- c("C_1","C_2","C_3","N_1","N_2")
# 计算RPKM
get_rpkm <- function(counts,libsize){
  rkm <- counts[,7:ncol(counts)]/counts$Length*1000
  rpkm <- data.frame(t(t(rkm) / libsize) * 10^6 )
  return(rpkm)
}
fread_c(fc_output_file)  -> rna_counts
# 确保libsize样本的顺序和counts中样本的顺序是一致的【已确认】
all(libsize$Sample==colnames_organized_1)
get_rpkm(rna_counts,libsize$libsize) -> rpkm
colnames(rpkm) <- colnames_organized_1
head(rpkm)

# 核内三个重复以及核外两个重复之间的相关性都很好
cor(rpkm, method = "pearson")

## 合并
mutate(rpkm,C=(C_1+C_2+C_3)/3,N=(N_1+N_2)/2) %>% dplyr::select(C,N) -> rpkm_N_C
head(rpkm_N_C)

# 导出基因的表达信息
rpkm_N_C %>% mutate(A=(N+C)/2) -> rpkm_N_C_A
rpkm_N_C_A$Geneid <- rna_counts$Geneid
fwrite_c(rpkm_N_C_A %>% select(Geneid,N,C,A), out_file)

head(rpkm_N_C_A)
nrow(rpkm_N_C_A)
filter(rpkm_N_C_A,A>0) -> human_21pcw_brain_expressed_gene_expr
nrow(human_21pcw_brain_expressed_gene_expr)
nrow(human_21pcw_brain_expressed_gene_expr)/nrow(rpkm_N_C_A)
# fwrite_c(human_21pcw_brain_expressed_gene_expr, out_file)

Sample,libsize
<chr>,<int>
p21_C_1,37163507
p21_C_2,40024668
p21_C_3,509442322
p21_N_1,46921455
p21_N_2,651581033


[1] "/home/user/data3/lit/project/sORFs/06-RNA-seq/02-output-custom-fa-gtf-20251022/mapping/p21_C_1"
[2] "/home/user/data3/lit/project/sORFs/06-RNA-seq/02-output-custom-fa-gtf-20251022/mapping/p21_C_2"
[3] "/home/user/data3/lit/project/sORFs/06-RNA-seq/02-output-custom-fa-gtf-20251022/mapping/p21_C_3"
[4] "/home/user/data3/lit/project/sORFs/06-RNA-seq/02-output-custom-fa-gtf-20251022/mapping/p21_N_1"
[5] "/home/user/data3/lit/project/sORFs/06-RNA-seq/02-output-custom-fa-gtf-20251022/mapping/p21_N_2"

[1] FALSE

,C_1,C_2,C_3,N_1,N_2
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,2.252444e+05,2.224842e+05,2.247718e+05,9.903268e+03,9.933972e+03
2,9.523465e-01,1.093277e+00,9.827285e-01,3.702894e-01,4.305931e-01
3,8.428259e+00,8.267296e+00,8.370865e+00,9.091133e-01,8.934658e-01
4,1.309397e-02,0.000000e+00,3.438711e-02,1.037091e-02,8.215092e-03
5,1.997462e+00,1.610950e+00,1.701393e+00,1.673336e+00,1.545318e+00
6,2.398869e+00,2.411756e+00,2.347374e+00,1.976502e+00,2.160374e+00


,C_1,C_2,C_3,N_1,N_2
C_1,1.0000000,0.9995331,0.9994938,0.3074274,0.3054358
C_2,0.9995331,1.0000000,0.9999973,0.3030622,0.3010637
C_3,0.9994938,0.9999973,1.0000000,0.3047467,0.3027492
N_1,0.3074274,0.3030622,0.3047467,1.0000000,0.9999968
N_2,0.3054358,0.3010637,0.3027492,0.9999968,1.0000000


,C,N
,<dbl>,<dbl>
1,2.241668e+05,9.918620e+03
2,1.009451e+00,4.004412e-01
3,8.355473e+00,9.012895e-01
4,1.582703e-02,9.292999e-03
5,1.769935e+00,1.609327e+00
6,2.385999e+00,2.068438e+00


,C,N,A,Geneid
,<dbl>,<dbl>,<dbl>,<chr>
1,2.241668e+05,9.918620e+03,1.170427e+05,7SK
2,1.009451e+00,4.004412e-01,7.049459e-01,A1BG
3,8.355473e+00,9.012895e-01,4.628381e+00,A2M
4,1.582703e-02,9.292999e-03,1.256001e-02,A4GALT
5,1.769935e+00,1.609327e+00,1.689631e+00,AAAS
6,2.385999e+00,2.068438e+00,2.227219e+00,AACS


[1] 19133

[1] 18479

[1] 0.9658182

In [14]:
libsize$Sample
colnames_organized_1

[1] "p21_C_1" "p21_C_2" "p21_C_3" "p21_N_1" "p21_N_2"

[1] "C_1" "C_2" "C_3" "N_1" "N_2"

In [3]:
nrow(filter(rpkm_N_C_A,C>=0.2))/nrow(rpkm_N_C_A)

[1] 0.7486019

In [4]:
summary(rpkm_N_C_A$C)
summary(rpkm_N_C_A$A)
summary(rpkm_N_C_A$N)

    Min.  1st Qu.   Median     Mean  3rd Qu.     Max. 
     0.0      0.2      1.1     92.5      3.2 768376.3 

    Min.  1st Qu.   Median     Mean  3rd Qu.     Max. 
     0.0      0.3      1.2     47.3      2.8 384629.0 

    Min.  1st Qu.   Median     Mean  3rd Qu.     Max. 
   0.000    0.248    0.935    2.164    2.090 9918.620 